In [1]:
!pip install evaluate seqeval datasets transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=7073b9e6fd2d8c4e69975a7e6be218d5847cfb02bec2e9bb0a65e31280dc7111
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


Import Library & Setup Awal

In [2]:
import torch
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import load_dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cu128
CUDA Available: True


Load & Eksplorasi Dataset

In [3]:
from datasets import load_dataset, DatasetDict

raw_datasets_temp = load_dataset("json", data_files="dataset_rekrutmen_sintetis.json")

# Membelah 500 data menjadi 80% Train dan 20% Validation
split_data = raw_datasets_temp["train"].train_test_split(test_size=0.2, seed=42)

# Susun ulang ke dalam struktur DatasetDict agar sesuai standar Hugging Face
raw_datasets = DatasetDict({
    "train": split_data["train"],
    "validation": split_data["test"]
})

label_list = [
    "O",
    "B-NAMA", "I-NAMA",
    "B-INSTITUSI", "I-INSTITUSI",
    "B-KETERAMPILAN", "I-KETERAMPILAN",
    "B-JABATAN", "I-JABATAN"
]

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print("Jumlah Data Train:", len(raw_datasets["train"]))
print("Jumlah Data Validation:", len(raw_datasets["validation"]))
print("\nContoh Teks Mentah (Indeks 0):")
print(raw_datasets["train"][0]["tokens"])

Generating train split: 0 examples [00:00, ? examples/s]

Jumlah Data Train: 400
Jumlah Data Validation: 100

Contoh Teks Mentah (Indeks 0):
['Saya', 'Natalia', 'Lestari', 'memiliki', 'pengalaman', 'sebagai', 'Frontend', 'Developer', 'dengan', 'keahlian', 'utama', 'Machine', 'Learning']


Inisialisasi Tokenizer & Model Transformer

In [4]:
model_checkpoint = "indobenchmark/indobert-base-p2"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

print(f"Model {model_checkpoint} berhasil dimuat dengan {len(label_list)} label klasifikasi.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You passed `num_labels=9` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model indobenchmark/indobert-base-p2 berhasil dimuat dengan 9 label klasifikasi.


Fungsi Alignment Token & Eksekusi Mapping

In [6]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    # Looping melalui setiap baris data
    for i, ner_tag_list in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Token spesial ([CLS], [SEP]) diberi label -100 agar diabaikan saat loss calculation
            if word_idx is None:
                label_ids.append(-100)
            # Beri label pada awal kata
            elif word_idx != previous_word_idx:
                # PERBAIKAN DI SINI: Ubah string (misal "B-NAMA") menjadi angka ID (misal 1)
                label_string = ner_tag_list[word_idx]
                label_ids.append(label2id[label_string])
            # Subword lanjutan diberi -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Mapping ke seluruh dataset
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print("\nContoh Token Setelah Alignment & Konversi ke ID (Indeks 0):")
tokens = tokenizer.convert_ids_to_tokens(tokenized_datasets["train"][0]["input_ids"])
labels = tokenized_datasets["train"][0]["labels"]
for token, label in zip(tokens, labels):
    print(f"{token:<15} : {label}")


Contoh Token Setelah Alignment & Konversi ke ID (Indeks 0):
[CLS]           : -100
saya            : 0
natal           : 1
##ia            : -100
lestari         : 2
memiliki        : 0
pengalaman      : 0
sebagai         : 0
front           : 7
##end           : -100
developer       : 8
dengan          : 0
keahlian        : 0
utama           : 0
machine         : 5
learning        : 6
[SEP]           : -100


Setup Metrik Evaluasi

In [7]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("Metrik Seqeval berhasil dimuat.")

Metrik Seqeval berhasil dimuat.


Eksekusi Fine-Tuning

In [8]:
args = TrainingArguments(
    output_dir="./indobert-ner-rekrutmen",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    # Argumen 'tokenizer' dihapus di sini
    compute_metrics=compute_metrics
)

print("Memulai proses fine-tuning Transformer...")
trainer.train()

# Menyimpan model dan tokenizer secara eksplisit
trainer.save_model("indobert-ner-rekrutmen-final")
tokenizer.save_pretrained("indobert-ner-rekrutmen-final")
print("Selesai! Model dan Tokenizer berhasil disimpan di folder 'indobert-ner-rekrutmen-final'.")

Memulai proses fine-tuning Transformer...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.071855,0.883721,0.869281,0.876442,0.972265
2,No log,0.022759,0.960000,0.941176,0.950495,0.990755
3,No log,0.012878,0.983553,0.977124,0.980328,0.996148
4,No log,0.008911,0.993399,0.983660,0.988506,0.997689
5,No log,0.007840,0.993421,0.986928,0.990164,0.998459


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai! Model dan Tokenizer berhasil disimpan di folder 'indobert-ner-rekrutmen-final'.
